# E2 – Forward Pass from Scratch
**Introduction to Deep Learning, THWS**

In this session you will build a neural network forward pass from the ground up — starting from a single scalar function and working your way up to a fully vectorised two-layer network.

**How to work:** All your implementations go in `linear.py`. Edit that file in your editor, then run the cells here to test and visualise your work. You do not need to modify this notebook.

**Important:** Use only basic PyTorch tensor operations (`torch.tensor`, `torch.dot`, `@`, `torch.clamp`, etc.). Do not use `torch.nn` or any other high-level PyTorch libraries — the point is to build everything from scratch.

**Structure:**
1. Scalar linear function and scalar ReLU
2. Shallow neural network (scalar input)
3. Vector input, scalar output
4. Batching: loop vs. vectorised
5. GPU acceleration
6. Two-layer MLP

In [ ]:
# reloads modules before executing
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import torch
import matplotlib.pyplot as plt

from linear import *
import plotting
import checks

---
## Block 1 – Scalar linear function and scalar ReLU

The simplest possible setting: $x \in \mathbb{R}, \ y \in \mathbb{R}$

**Task 1.1** Implement the linear function $f_\theta(x) = \theta_1 \cdot x + \theta_0$, where $\theta = (\theta_0, \theta_1)$ as `linear_scalar(x, theta)` in `linear.py` - *(slide 6 of Session 2)*.  

**Task 1.2** Implement the ReLU $y = ReLU(x) = \max(0, x)$ as `relu_scalar(x)` in `linear.py` - *(slide 6 of Session 2)*.    

In [ ]:
# sanity checks
checks.check_block1()

### Plotting

In [ ]:
# plot linear
theta = torch.tensor([-1.0, 3.0])

fig = plotting.plot_func(func=linear_scalar, theta=theta, save_path='figures/linear')

**Task 1.1** plot linear functions for different parameter values and make sure you understand their meaning.

***Question:*** For $\bf\theta = (-1.0, 2.0)$ at which point the function $f_\theta$ 
- intersects the $x$ axis? ***Answer:*** here
- intersects the $y$ axis? ***Answer:*** here

***Question:*** For which parameters $\bf\theta = (\theta_0, \theta_1)$ the function $f_\theta$ intersects the $x$ axis at point 3 and $y$ axis at point 2?  
***Answer:*** $\theta = (., .)$

In [ ]:
# plot relu
fig = plotting.plot_func(func=relu_scalar, save_path='figures/relu')

In [ ]:
# close plots
plt.close()

---
## Block 2 – Shallow Neural Network (scalar input)

A single ReLU unit produces one hinge. What happens when we combine multiple?

**Task 2.1** Implement `relu_unit(x, theta)` in `linear.py` - *(slide 7 of Session 2)*.  .

**Task 2.2** Implement `shallow(x, theta_hidden, theta_out)` in `linear.py` - *(slide 9 of Session 2)*.  
as a weighted sum of ReLU units via a loop.

In [ ]:
# sanity checks
checks.check_block2()

### Plotting

In [ ]:
# ReLu unit
theta=torch.tensor([-0.5, 2.0])

fig = plotting.plot_func(func=relu_unit, theta=theta, save_path='figures/relu_unit')

**Task 2.3** Plot a single ReLU unit for several values of $\theta$ and answer:
- What is the effect of $\theta_1$ on the output?  
  ***Answer:*** here
- What is the effect of $\theta_0$ on the output?  
  ***Answer:*** here
- At which point does the bend occur?  
  ***Answer:*** here

In [ ]:
# Shallow network with 2 hidden units
theta_hidden = [torch.tensor([-0.5, 2.0]), torch.tensor([1.0, -1.5])]
theta_out = torch.tensor([0.0, 1.0, -1.0])

theta = [theta_hidden, theta_out]

fix = plotting.plot_func(func=shallow, theta=[theta_hidden, theta_out], label="2 units", save_path='figures/shallow_2')

**Task 2.4** Plot the `shallow` network for 1, 2, and 5 hidden units of your choice and answer:
- What is the effect of adding more hidden units?  
  ***Answer:*** here
- Is the output always piecewise linear, regardless of the number of units?  
  ***Answer:*** here
- Can you find parameters that produce a constant output? If yes, how?  
  ***Answer:*** here

---
## Block 3 – Vector input, scalar output

Real data is rarely one-dimensional. Let's generalise to $\mathbf{x} \in \mathbb{R}^d, \ y \in \mathbb{R}$ - *(slide 11 of Session 2)*.

**Task 3.1** Implement `linear_vector(x, theta)` in `linear.py` using `torch.dot`.

**Task 3.2** Implement `relu_tensor(x)` in `linear.py` using `torch.clamp`.

**Task 3.3** Implement `relu_unit_vector(x, theta)` in `linear.py`.

**Task 3.4** Implement `shallow_vector(x, theta_hidden, theta_out)` in `linear.py`.

In [ ]:
# sanity checks
checks.check_block3()

### Plotting

In [ ]:
#Linear and ReLU plots
theta = torch.tensor([0.0, 1.0, 0.5])   # theta_0=0, theta_1=[1.0, 0.5]
fig = plotting.plot_surface(func=linear_vector, theta=theta,
                      title="Linear: vector input", save_path='figures/linear_vector')

fig = plotting.plot_surface(func=relu_unit_vector, theta=theta,
                      title="ReLU unit: vector input", save_path='figures/relu_unit_vector')

**Task 3.5** Plot `linear_vector` and `relu_unit_vector` as a surface for a 2D input of your choice and answer:
- What does the linear surface look like geometrically?  
  ***Answer:*** here
- What changes after applying ReLU?  
  ***Answer:*** here

In [ ]:
# Shallow_vector with 1 and 3 hidden units
theta_hidden_1 = [torch.tensor([0.0, 1.0, 0.5])]
theta_out_1 = torch.tensor([0.0, 1.0])

fig = plotting.plot_surface(func=shallow_vector, theta=[theta_hidden_1, theta_out_1],
                      title="Shallow: 1 hidden unit", save_path='figures/shallow_vector_1')


**Task 3.6** Plot `shallow_vector` for 1 and 3 hidden units and answer:
- How does the surface change compared to a single ReLU unit?  
  ***Answer:*** here

----
## Block 4 – Batching: loop vs. vectorised

So far we process one sample at a time. In practice we always have $N$ samples $X \in \mathbb{R}^{N \times d}$.

**Task 4.1** Implement `shallow_batch_loop(X, theta_hidden, theta_out)` in `linear.py`
— loop over rows of $X$, call `shallow_vector` for each one.

In [ ]:
checks.check_block4_1()

**Task 4.2** Time it for a large $N$ and observe. What is the shape of the output? How does it scale with $N$?  
***Answer:*** here

In [ ]:
# timing – loop over samples
N = 100000
X = torch.randn(N, 2)
theta_hidden = [torch.tensor([0.0, 1.0, 0.5]), torch.tensor([0.5, -1.0, 1.0])]
theta_out = torch.tensor([0.0, 1.0, -1.0])

t0 = time.time()
out_loop = shallow_batch_loop(X, theta_hidden, theta_out)
t_loop = time.time() - t0
print(f"Loop time: {t_loop:.4f} s")

**Task 4.3** Let's try without looping over examples. First implement `linear_batch(X, theta)` in `linear.py` — vectorise `linear_vector` over $N$ samples using the `@` operator - it shall in one go produce output for all lines in $X$. Then implement `shallow_batch(X, theta_hidden, theta_out)` using `linear_batch` wihtout looping over the samples.

In [ ]:
checks.check_block4_3()

**Task 4.4** Time `shallow_batch` and answer:
- How much faster is the vectorised version?  
  ***Answer:*** here
- Why is the loop slow?  
  ***Answer:*** here

In [ ]:
# timing – vectorised over samples
t0 = time.time()
out_vec = shallow_batch(X, theta_hidden, theta_out)
t_vec = time.time() - t0
print(f"Loop time:       {t_loop:.4f} s")
print(f"Vectorised time: {t_vec:.4f} s")
print(f"Speed-up:        {t_loop / t_vec:.1f}x")
assert torch.allclose(out_loop, out_vec, atol=1e-5), "Outputs do not match!"

**Task 4.5** Above, we still had one more loop - over the hidden units. Let's get rid of it. First implement `linear_layer(X, Theta)` in `linear.py` — a linear layer mapping a batch $X \in \mathbb{R}^{N \times d}$ to multiple outputs $\in \mathbb{R}^{N \times k}$, where $k$ is the number of hidden units.

Before implementing, answer:
- What is the shape of `Theta`?  
  ***Answer:*** here
- What operation produces all $k$ outputs for all $N$ samples simultaneously?  
  ***Answer:*** here
- What is the shape of the output of `linear_layer`?  
  ***Answer:*** here

Then implement `shallow_batch_vectorised(X, Theta_hidden, theta_out)` using `linear_layer`.

In [ ]:
checks.check_block4_5()

**Task 4.6** Time this version and answer:
- Is the fully vectorised version faster?  
  ***Answer:*** here

In [ ]:
# timing – fully vectorised
Theta_hidden = torch.stack([torch.tensor([0.0, 1.0, 0.5]), torch.tensor([0.5, -1.0, 1.0])])

t0 = time.time()
out_full = shallow_batch_vectorised(X, Theta_hidden, theta_out)
t_full = time.time() - t0
print(f"Loop time:            {t_loop:.4f} s")
print(f"Vectorised (samples): {t_vec:.4f} s")
print(f"Fully vectorised:     {t_full:.4f} s")
assert torch.allclose(out_loop, out_full, atol=1e-5), "Outputs do not match!"

**Task 4.7**
- Does the speed of the fully vectorised version depend on $k$?  
  ***Answer:*** here
- On what hardware would you expect the biggest speedup from full vectorisation?  
  ***Answer:*** here

---
## Block 5 – GPU Acceleration

So far everything ran on CPU. Modern deep learning relies on GPUs for massive parallelism. Let's move our computation to GPU and observe the speedup.

Note: if you do not have a GPU available, `device` will fall back to CPU and you will not see a speedup — but the code will still run correctly.

In [ ]:
# device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on: {device}')

In PyTorch, every tensor lives on a specific device — either CPU or GPU. By default all tensors are created on CPU. To use the GPU, we need to explicitly move tensors there using `.to(device)`.

Computation can only happen between tensors on the **same device** — mixing CPU and GPU tensors in one operation will raise an error. This means we need to move both the data $X$ and all parameters to GPU before running the forward pass.

Under the hood, `.to(device)` allocates new memory on the GPU and copies the tensor data there. The original CPU tensor is unchanged.

In [ ]:
# move data and parameters to device
N = 100_000
X_gpu = torch.randn(N, 2).to(device)
Theta_hidden_gpu = torch.stack([torch.tensor([0.0, 1.0, 0.5]),
                                 torch.tensor([0.5, -1.0, 1.0])]).to(device)
theta_out_gpu = torch.tensor([0.0, 1.0, -1.0]).to(device)

# warm-up (first GPU call includes compilation overhead)
_ = shallow_batch_vectorised(X_gpu, Theta_hidden_gpu, theta_out_gpu)

t0 = time.time()
out_gpu = shallow_batch_vectorised(X_gpu, Theta_hidden_gpu, theta_out_gpu)
t_gpu = time.time() - t0

print(f"Fully vectorised (CPU): {t_full:.4f} s")
print(f"Fully vectorised (GPU): {t_gpu:.4f} s")
print(f"Speed-up:               {t_full / t_gpu:.1f}x")

**Task 5.1** Answer:
- Experiment with increasing $k$ (number of hidden units). At what point does the GPU become significantly faster than CPU?  
  ***Answer:*** here
- Why does the GPU help for the fully vectorised version but not for the loop version?  
  ***Answer:*** here
- Comment out the warm-up call and re-run the timing. What do you observe? Why?  
  ***Answer:*** here

In [ ]:
# experiment with increasing k
for k in [2, 10, 50, 100, 500, 1000]:
    Th_cpu = torch.randn(k, 3)
    Th_gpu = Th_cpu.to(device)
    th_out_cpu = torch.randn(k + 1)
    th_out_gpu = th_out_cpu.to(device)

    t0 = time.time(); shallow_batch_vectorised(X, Th_cpu, th_out_cpu); t_cpu = time.time() - t0
    _ = shallow_batch_vectorised(X_gpu, Th_gpu, th_out_gpu)  # warm-up
    t0 = time.time(); shallow_batch_vectorised(X_gpu, Th_gpu, th_out_gpu); t_gpu = time.time() - t0
    print(f"k={k:5d}:  CPU={t_cpu:.4f}s  GPU={t_gpu:.4f}s  speed-up={t_cpu/t_gpu:.1f}x")

---
## Block 6 – Two-layer MLP

So far we have built a shallow network with one hidden layer. A deep network stacks multiple hidden layers — each layer's output becomes the next layer's input.

**Task 6.1** Implement `mlp_batch(X, Theta_1, Theta_2, theta_out)` in `linear.py`  
— extend `shallow_batch_vectorised` by passing the hidden activations $H_1$ through a second hidden layer before the output.

Before implementing, answer:
- What is the shape of $H_1$?  
  ***Answer:*** here
- What is the shape of $\Theta_2$?  
  ***Answer:*** here
- What is the shape of $H_2$?  
  ***Answer:*** here

In [ ]:
checks.check_block6()

**Task 6.2** Run the forward pass and inspect the output:
- What changes compared to `shallow_batch_vectorised`?  
  ***Answer:*** here

In [ ]:
# two-layer MLP forward pass
N, d, k1, k2 = 100, 2, 8, 4
X = torch.randn(N, d)
Theta_1   = torch.randn(k1, d + 1)
Theta_2   = torch.randn(k2, k1 + 1)
theta_out = torch.randn(k2 + 1)

out = mlp_batch(X, Theta_1, Theta_2, theta_out)
print(f"Input shape:  {X.shape}")
print(f"Output shape: {out.shape}")